In [6]:
from typing import List, TypedDict, Annotated
from langchain_google_genai import ChatGoogleGenerativeAI

import operator

class PlanExecuteState(TypedDict):
    input: str # User's original query
    plan: List[str] # The steps to follow
    past_steps: Annotated[List[tuple], operator.add] # (Step, Result) pairs
    response: str # Final answer
    

In [7]:

from pydantic import BaseModel, Field

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)
# 1. Define the structure for the plan 
class Plan(BaseModel):
    """Plan to follow in future"""
    steps: List[str] = Field( description= "different steps to follow, should be in sorted order")
# 2. Define the Planner Logic
def planner_node(state: PlanExecuteState):
    """We ask Gemini to output structured data matching the 'Plan' class"""
    planner = llm.with_structured_output(Plan)
    
    plan = planner.invoke(
        [SystemMessage(content= "You are a planner. Generate a step-by-step plan for the user's task."),
         HumanMessage(content = state["input"])
         ]
    )
    
    # Return the plan to the state
    return {"plan": plan.steps}

In [8]:
# We reuse the tools you already defined

def executor_node(state: PlanExecuteState):
    # Get the plan 
    plan = state["plan"]
    
    # Get the next step (first item in the plan)
    current_step = plan[0]
    
    # Execute the step using a simple tool-calling LLM call
    # We bind tools just for this execution
    
    llm_with_tools = llm.bind_tools(tools)
    
    # We ask Gemini to executte THIS specific step
    messages = [
        SystemMessage(content = "You are a helpful executor. Execute the user's step using tools"),
        HumanMessage(content=current_step)
    ]
    
    result = llm_with_tools.invoke(messages)
    
    # If it called a tool, we need to run it (Simplified for clarity: normally we'd use ToolNode)
    # For this pattern, let's assume the ReAct agent handles the execution 
    # In a full production setup, we would call the 'agent_manual' we built earlier here!
    
    return {
        "past_steps": [(current_step, result.content)], # Save result
        "plan": plan[1:] # Remove the executed step
    }

In [ ]:
from langgraph.graph import StateGraph, START, END

# Logic to check if we are done

def should_end(state: PlanExecuteState):
    if len(state["plan"]) > 0:
        return "executor" # Still have steps? Keep executing.
    return "response" # No steps? Go to final response.

# Final response Node
def response_node(state: PlanExecuteState):
    # Ask Gemini to summarize all past steps into a final answer
    context = "\n".join([f"Step: {s}\nResult: {r}" for s, r in state["past_steps"]])
    response = llm.invoke(f"Based on these results, answer the user:\n{context}")
    return {"response": response.content}
# --- BUILD GRAPH ---

workflow = StateGraph(PlanExecuteState)

workflow.add_node("planner", planner_node)
workflow.add_node("executor_node", executor_node)
workflow.add_node("response", response_node)

# Add Edges
workflow.add_edge(START, "planner")
workflow.add_edge("planner", "executor")

# Conditional Edge: Loop or finish
workflow.add_conditional_edges(
    "executor",
    should_end,
    {
        "executor": "executor",
        "response": "response"
    }
)

workflow.add_edge("response", END)

# Compile

plan_execute_agent = workflow.compile()
print("Plan-Execute Agent Created")
